# Тестирование задач 13-15

Этот ноутбук тестирует реализацию оценки входящих данных для optimizer на основе исторических β.

**Задача 13**: Рассчитать ковариационную матрицу на основе исторических β

**Задача 14**: Построить границу эффективных портфелей на основе полученной ковариационной матрицы

**Задача 15**: Построить границу эффективных портфелей для разных исторических окон

Выбор из задач 11-12:
- Индекс: IMOEX (Мосбиржа)
- Окно: Скользящее окно длиной в 1 год (252 торговых дня)
- Схема взвешивания: Равные веса наблюдений

In [ ]:
import sys
sys.path.insert(0, '.')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from task_13_14_15 import (
    calculate_market_model_betas,
    calculate_all_betas,
    calculate_covariance_from_betas,
    calculate_residual_variances,
    task_13_covariance_from_historical_betas,
    task_14_efficient_frontier_from_betas,
    task_15_efficient_frontier_dynamics_betas,
    compare_covariance_methods
)

# Настройка графиков
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
%matplotlib inline

pd.set_option('display.max_columns', 15)
pd.set_option('display.width', 120)

## 1. Загрузка данных

In [ ]:
# Импорт функций загрузки из task_2_3
sys.path.insert(0, '../task-2-3')
from task_2_3 import load_prices_data, calculate_returns

# Загрузка данных
prices = load_prices_data('../data/prices_moex_new.csv')

print("=== Информация о данных ===")
print(f"Размер: {prices.shape}")
print(f"Период: {prices.index.min()} - {prices.index.max()}")
print(f"Количество акций: {len(prices.columns)}")
print(f"\nСписок акций: {list(prices.columns)}")

## 2. Расчет доходностей

In [ ]:
# Расчет логарифмических доходностей
returns = calculate_returns(prices)

print("=== Информация о доходностях ===")
print(f"Размер: {returns.shape}")
print(f"Период: {returns.index.min()} - {returns.index.max()}")
print(f"\nПервые 5 строк:")
print(returns.head())

## 3. Тестирование рыночной модели

In [ ]:
# Проверка функции расчета бета для одной акции
print("=== Тестирование рыночной модели ===")

# Используем первую акцию и индекс MOEX
test_ticker = returns.columns[0]
market_ticker = 'MOEX'

alpha, beta = calculate_market_model_betas(returns[test_ticker], returns[market_ticker])

print(f"Акция: {test_ticker}")
print(f"Рыночный индекс: {market_ticker}")
print(f"α (альфа): {alpha:.6f}")
print(f"β (бета): {beta:.6f}")

# Интерпретация бета
if beta > 1:
    beta_interp = "агрессивная (более волатильна, чем рынок)"
elif beta > 0:
    beta_interp = "защитная (менее волатильна, чем рынок)"
elif beta < 0:
    beta_interp = "противоположная рынку"
else:
    beta_interp = "без корреляции с рынком"

print(f"Интерпретация: {beta_interp}")

## 4. Расчет бета для всех акций

In [ ]:
# Расчет бета для всех акций относительно индекса MOEX
print("=== Расчет бета для всех акций ===")

betas = calculate_all_betas(returns, market_ticker='MOEX')

print(f"\nРассчитано {len(betas)} бета-коэффициентов")
print(f"\nБета-коэффициенты (все акции):")
print(betas)

# Статистика
print(f"\n=== Статистика бета ===")
print(f"Среднее: {betas['beta'].mean():.4f}")
print(f"Стд. отклонение: {betas['beta'].std():.4f}")
print(f"Минимум: {betas['beta'].min():.4f} ({betas['beta'].idxmin()})")
print(f"Максимум: {betas['beta'].max():.4f} ({betas['beta'].idxmax()})")

# Классификация по бета
defensive = betas[betas['beta'] < 1].index
aggressive = betas[betas['beta'] > 1].index

print(f"\nЗащитные акции (β < 1): {len(defensive)}")
print(f"Агрессивные акции (β > 1): {len(aggressive)}")

## 5. Визуализация бета

In [ ]:
# График бета-коэффициентов
plt.figure(figsize=(14, 6))

colors = ['red' if b < 1 else 'green' for b in betas['beta']]
plt.bar(range(len(betas)), betas['beta'], color=colors, alpha=0.7)
plt.axhline(y=1, color='blue', linestyle='--', linewidth=2, label='β = 1 (рынок)')
plt.xticks(range(len(betas)), betas.index, rotation=90)
plt.xlabel('Акция')
plt.ylabel('Бета-коэффициент')
plt.title('Бета-коэффициенты всех акций относительно индекса MOEX')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Задача 13: Ковариационная матрица на основе исторических β

In [ ]:
# Задача 13: Расчет ковариационной матрицы на основе исторических β
print("=== Задача 13 ===")

betas_result = task_13_covariance_from_historical_betas(
    returns, market_ticker='MOEX', include_residuals=True
)

In [ ]:
# Сравнение ковариационных матриц
print("\n=== Сравнение ковариационных матриц ===")

stock_tickers = [col for col in returns.columns if col != 'MOEX']

cov_classic = returns[stock_tickers].cov()
cov_beta = pd.DataFrame(
    betas_result['cov_matrix'],
    index=stock_tickers,
    columns=stock_tickers
)

print(f"\nКлассическая ковариационная матрица (первые 5x5):")
print(cov_classic.iloc[:5, :5])

print(f"\nКовариационная матрица на основе β (первые 5x5):")
print(cov_beta.iloc[:5, :5])

# Разность
diff = (cov_classic - cov_beta).abs()
print(f"\nАбсолютная разность (первые 5x5):")
print(diff.iloc[:5, :5])
print(f"\nСредняя разность: {diff.mean().mean():.6f}")

## 7. Задача 14: Эффективная граница на основе β

In [ ]:
# Задача 14: Эффективная граница на основе ковариационной матрицы из β
print("=== Задача 14 ===")

stock_returns = returns[stock_tickers]
mean_returns = stock_returns.mean().values

ef_returns, ef_stds = task_14_efficient_frontier_from_betas(
    betas_result['cov_matrix'],
    mean_returns,
    n_points=50,
    method_name="Исторические β"
)

In [ ]:
# Визуализация эффективной границы
plt.figure(figsize=(10, 6))
plt.plot(ef_stds, ef_returns, 'b-', linewidth=2, label='На основе исторических β')
plt.scatter(ef_stds, ef_returns, c=ef_returns, cmap='viridis', s=20)
plt.colorbar(label='Доходность')
plt.xlabel('Риск (стандартное отклонение)')
plt.ylabel('Доходность')
plt.title('Эффективная граница на основе исторических β')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Задача 15: Динамика эффективных границ на основе β

In [ ]:
# Задача 15: Динамика эффективных границ на основе β
print("=== Задача 15 ===")

beta_frontiers, beta_stability = task_15_efficient_frontier_dynamics_betas(
    returns, market_ticker='MOEX', window_size='1Y', step_size='1Y',
    include_residuals=True, n_points=50
)

In [ ]:
# Метрики стабильности
print("=== Метрики стабильности ===")
print(beta_stability)

In [ ]:
# Визуализация эффективных границ для всех окон
plt.figure(figsize=(14, 8))
colors = plt.cm.viridis(np.linspace(0, 1, len(beta_frontiers)))

for i, (date, frontier) in enumerate(sorted(beta_frontiers.items())):
    label = date.strftime('%Y-%m-%d') if i % 2 == 0 or i == len(beta_frontiers) - 1 else None
    plt.plot(frontier['stds'], frontier['returns'], 
             color=colors[i], linewidth=2, alpha=0.7, label=label)

plt.xlabel('Риск (стандартное отклонение)')
plt.ylabel('Доходность')
plt.title('Динамика эффективных границ на основе исторических β (скользящее окно 1 год)')
plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# График динамики минимального риска во времени
plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
plt.plot(beta_stability.index, beta_stability['min_std'], marker='o', linewidth=2)
plt.title('Минимальный риск (GMVP) во времени (на основе β)')
plt.xlabel('Дата')
plt.ylabel('Стандартное отклонение')
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)

plt.subplot(1, 2, 2)
plt.plot(beta_stability.index, beta_stability['min_std_return'], marker='s', linewidth=2, color='orange')
plt.title('Доходность GMVP во времени (на основе β)')
plt.xlabel('Дата')
plt.ylabel('Доходность')
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# График динамики максимального Шарпа во времени
plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
plt.plot(beta_stability.index, beta_stability['max_sharpe'], marker='o', linewidth=2)
plt.title('Максимальное Шарп-отношение во времени (на основе β)')
plt.xlabel('Дата')
plt.ylabel('Шарп-отношение')
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)

plt.subplot(1, 2, 2)
plt.plot(beta_stability.index, beta_stability['efficiency_ratio'], marker='s', linewidth=2, color='green')
plt.title('Коэффициент эффективности во времени (на основе β)')
plt.xlabel('Дата')
plt.ylabel('Коэффициент эффективности')
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

## 9. Сравнение методов расчета ковариационных матриц

In [ ]:
# Сравнение трех методов
print("=== Сравнение методов расчета ковариационных матриц ===")

comparison = compare_covariance_methods(returns, market_ticker='MOEX')

In [ ]:
# Визуальное сравнение тепловых карт ковариационных матриц
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

methods = [('Классическая', comparison['classic']), 
           ('Бета (без остатков)', comparison['beta_simple']),
           ('Бета (с остатками)', comparison['beta_residuals'])]

for idx, (title, matrix) in enumerate(methods):
    sns.heatmap(matrix[:10, :10], ax=axes[idx], annot=True, fmt='.4e',
                cmap='coolwarm', center=0, cbar_kws={'label': 'Ковариация'})
    axes[idx].set_title(title)
    axes[idx].set_xticklabels(stock_tickers[:10], rotation=90)
    axes[idx].set_yticklabels(stock_tickers[:10])

plt.tight_layout()
plt.show()

## 10. Итоговый анализ

In [ ]:
print("=== Итоговый анализ ===")
print(f"\nПериод данных: {returns.index.min()} - {returns.index.max()}")
print(f"Всего наблюдений: {len(returns)}")
print(f"Количество акций: {len(stock_tickers)}")
print(f"Рыночный индекс: MOEX")

print(f"\n--- Бета-коэффициенты ---")
print(f"Средняя бета: {betas_result['betas'].mean():.4f}")
print(f"Стд. отклонение бета: {betas_result['betas'].std():.4f}")
print(f"Дисперсия рынка: {betas_result['market_variance']:.6f}")

print(f"\n--- Задача 15: Динамика ---")
print(f"Получено окон: {len(beta_frontiers)}")
print(f"Средний минимальный риск: {beta_stability['min_std'].mean():.6f}")
print(f"Средний макс. Шарп: {beta_stability['max_sharpe'].mean():.6f}")

print(f"\n--- Сравнение методов ---")
print(f"Классическая матрица - среднее: {comparison['classic'].mean():.6f}")
print(f"Бета (без остатков) - среднее: {comparison['beta_simple'].mean():.6f}")
print(f"Бета (с остатками) - среднее: {comparison['beta_residuals'].mean():.6f}")

## Вывод

В этом ноутбуке мы протестировали:
1. Расчет параметров рыночной модели (α и β)
2. Расчет бета-коэффициентов для всех акций относительно индекса MOEX
3. Задачу 13: Расчет ковариационной матрицы на основе исторических β
4. Задачу 14: Построение эффективной границы на основе β
5. Задачу 15: Динамику эффективных границ на основе β
6. Сравнение различных методов расчета ковариационных матриц

Все функции работают корректно и позволяют анализировать портфели с использованием рыночной модели.